# Qwen2.5-3B — RunPod · 4 GPUs · eight answers per problem

Use **one Linux RunPod Pod with four RTX PRO 6000 GPUs (96 GB each)** and JupyterLab.
Each GPU loads its own pinned Qwen2.5-3B base model and processes a disjoint subset
of problems. Each problem still receives eight responses. This is generation and
grading, not model training. No multi-node networking or tensor parallelism is used.

Before running:
- Attach persistent storage at `/workspace`. A **network volume** survives Pod
  deletion; a Pod volume disk does not. Check available storage for the runtime,
  model cache, response journals, merged journal, Parquet and grading audit.
- Supply a write-capable `HF_TOKEN` environment variable, or enter it into the hidden prompt.
- Run all cells. The notebook installs its embedded code and locked runtime;
  no repository clone is needed. GPU preflight checks precede full generation.

Sampling remains temperature=1, top-p=1, maximum 8,192 new tokens per answer,
with the same prompt, grading and quality filters as the Colab notebook.
The source tag `synthetic_math` no longer excludes a row; all other quality rules remain.
Use the new default RUN_NAME: earlier filtered runs have different row assignments.
Rows with 0/8 correct remain in the output. There is no generation smoke gate.
**UPLOAD_FULL=True publishes automatically to Seungjun/clean-math-sft-pool-30k
after all four workers finish and the merged result validates.**

Re-run using the same settings, code and RUN_NAME to resume completed problems.
Each worker has its own journal and log. Changed settings/code or GPU type are rejected.
Keep the Pod and Jupyter kernel running; this is not a detached background job.
Interrupting the generation cell stops workers; unfinished problems are regenerated on resume.
Do not run two copies against the same RUN_NAME or overwrite the code during generation.

Storage reference: [RunPod storage guide](https://www.runpod.io/blog/where-did-my-files-go-a-straight-guide-to-runpod-storage).

In [ ]:
RUN_NAME = 'qwen25-3b-base-eight-runpod4-other-v3-no-source-exclusion'
STORAGE_ROOT = '/workspace/LG-SFT-Pool-Sampling'
CODE_ROOT = '/workspace/lg-sft-pool-sampling-runpod4'
EVAL_ENV = '/workspace/lg-eval-env'
GPU_IDS = ['0', '1', '2', '3']  # CUDA GPU IDs or UUIDs visible to this Pod.
SOURCE_REVISION = 'main'  # First run pins this; resume reuses the saved commit.
# After an earlier upload, use that run's ORIGINAL dataset_revision instead of main.
TEMPERATURE = 1.0
TOP_P = 1.0
MAX_NEW_TOKENS = 8192
MAX_NUM_SEQS = 16  # Per GPU: conservative original setting, 64 active sequences total.
SEED = 42
UPLOAD_FULL = True

## 1. Install the isolated runtime and check all four GPUs

In [ ]:
import base64, getpass, io, json, os, subprocess, sys, zipfile
from pathlib import Path
if sys.platform != 'linux':
    raise RuntimeError('Run this notebook in the Linux RunPod Pod')
if len(GPU_IDS) != 4 or len(set(GPU_IDS)) != 4:
    raise ValueError('Four distinct GPUs are required')
os.environ['CUDA_VISIBLE_DEVICES'] = ','.join(GPU_IDS)
os.environ['HF_HOME'] = '/workspace/huggingface'
os.environ['HF_TOKEN'] = os.environ.get('HF_TOKEN') or getpass.getpass('Hugging Face write token: ')
if not os.environ['HF_TOKEN']:
    raise ValueError('HF_TOKEN is required')
Path(CODE_ROOT).mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode('...'))) as bundle:
    bundle.extractall(CODE_ROOT)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.11.22', 'PyYAML==6.0.3'])
subprocess.check_call([sys.executable, 'scripts/setup_eval_runtime.py', '--venv', EVAL_ENV], cwd=CODE_ROOT)
sys.path.insert(0, CODE_ROOT)
import importlib
import common.pool_progress as pool_progress
importlib.reload(pool_progress)
run_logged = pool_progress.run_pool_logged
gpu_check = """
import torch
assert torch.cuda.device_count() == 4, 'The four selected GPUs must all be visible'
for i in range(4):
    props = torch.cuda.get_device_properties(i)
    print(i, props.name, round(props.total_memory / 2**30, 1), 'GiB')
    assert props.total_memory >= 90 * 2**30, 'Expected a 96 GB GPU'
    x = torch.ones(1, device=f'cuda:{i}')
    assert x.item() == 1
"""
import textwrap
run_logged([str(Path(EVAL_ENV) / 'bin/python'), '-c', textwrap.dedent(gpu_check)],
           cwd=CODE_ROOT, log_path=Path(CODE_ROOT) / 'gpu_preflight.log')

## 2. Check grading and pin the cleaned input

Only the coordinator prepares the input and writes the shared manifest. The original
text cleanup and additional quality filters are unchanged. Inspect the printed row
count and quality report below. The full run partitions eligible rows by index modulo 4.

In [ ]:
run_logged([str(Path(EVAL_ENV) / 'bin/python'), '-m', 'eval.pool_grading_checks'],
           cwd=CODE_ROOT, log_path=Path(CODE_ROOT) / 'grading_checks.log')
import yaml
CONFIG = yaml.safe_load((Path(CODE_ROOT) / 'configs/sft_pool_sampling.yaml').read_text())
CONFIG['data'].update(revision=SOURCE_REVISION, expected_rows=None, require_text_cleanup=True)
CONFIG['sampling'].update(temperature=TEMPERATURE, top_p=TOP_P,
                          max_new_tokens=MAX_NEW_TOKENS, seed=SEED)
CONFIG['engine']['max_num_seqs'] = MAX_NUM_SEQS
CONFIG['smoke']['seed'] = SEED
RUN_DIR = Path(STORAGE_ROOT) / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = RUN_DIR / 'requested_config.yaml'
CONFIG_PATH.write_text(yaml.safe_dump(CONFIG, sort_keys=False))
COMMAND = [str(Path(EVAL_ENV) / 'bin/python'), '-m', 'pipeline.sample_sft_pool_runpod',
           '--config', str(CONFIG_PATH), '--run-dir', str(RUN_DIR), '--gpus', ','.join(GPU_IDS)]
subprocess.check_call([str(Path(EVAL_ENV) / 'bin/python'), '-m', 'common.pool_progress',
                       '--check-resume', str(CONFIG_PATH), str(RUN_DIR)], cwd=CODE_ROOT)
run_logged(COMMAND + ['--mode', 'prepare'], cwd=CODE_ROOT,
           log_path=RUN_DIR / 'prepare_console.log')
manifest = json.loads((RUN_DIR / 'manifest.json').read_text())
print('고정된 HF 버전:', manifest['dataset_revision'])
print('HF 입력 행 수:', manifest['source_rows'])
quality = json.loads((RUN_DIR / 'quality_summary.json').read_text())
print(f"Eligible problems: {quality['eligible_rows']:,} / {quality['input_rows']:,}")
print('Quality report:', RUN_DIR / 'quality_summary.json')
print('저장 위치:', RUN_DIR)

## 3. Run all four GPU workers, then merge

Each worker uses tensor_parallel_size=1 and its own CUDA_VISIBLE_DEVICES.
The model is downloaded once to the shared cache before workers launch.
A tqdm bar per GPU updates every 30 seconds with counts, elapsed time and estimated remaining time; detailed logs are in
`RUN_DIR/workers/0/console.log` through `RUN_DIR/workers/3/console.log`.
Startup compilation can take several minutes. CPU grading runs separately in each worker.

Each completed problem is synced to its worker journal. The coordinator stops the
other workers if one fails. Fix the cause and rerun with unchanged settings to resume;
tuning sampling/batching requires a new RUN_NAME. A merge is allowed only when every
expected row occurs exactly once in the correct worker, with valid annotations and
matching runtime/prompt records. Export keeps original input order.

In [ ]:
run_logged(COMMAND + ['--mode', 'full'], cwd=CODE_ROOT,
           log_path=RUN_DIR / 'full_console.log')
summary = json.loads((RUN_DIR / 'full/summary.json').read_text())
print(f"Completed: {summary['rows']:,} problems / {summary['responses']:,} responses")
print(f"Response accuracy: {summary['accuracy_per_response']:.2%} | At least one correct: {summary['at_least_one_correct_fraction']:.2%}")
print(f"Truncated: {summary['truncated_fraction']:.2%} | Grading exceptions: {summary['grading_exceptions']:,}")
print('Full summary:', RUN_DIR / 'full/summary.json')
print('전체 결과 Parquet:', RUN_DIR / 'full/data')
print('채점 상세 기록:', RUN_DIR / 'full/grading_audit.jsonl')

## 4. 완료된 전체 결과를 HF에 업로드

모든 대상 행의 생성이 끝나야 업로드할 수 있습니다.
**Seungjun/clean-math-sft-pool-30k**의 기본 train 데이터를 이번 결과로
갱신하며 기존 원본 파일은 보존합니다. 생성 도중 HF 데이터가 다른 버전으로
변경되었다면 덮어쓰지 않고 중단합니다. persistent storage의 결과는 유지됩니다.

In [ ]:
if UPLOAD_FULL:
    run_logged(COMMAND + ['--mode', 'upload'], cwd=CODE_ROOT,
               log_path=RUN_DIR / 'upload_console.log')
    receipt = json.loads((RUN_DIR / 'upload_receipt.json').read_text())
    print('업로드 및 검증 완료:', receipt['url'])
else:
    print('결과는 persistent storage에 저장되었습니다:', RUN_DIR / 'full/data')

## Files and restart notes

- `workers/<rank>/responses.jsonl`: durable per-GPU checkpoints; retain all four.
- `workers/<rank>/console.log`: startup, generation progress and failure details.
- `parallel_protocol.json`: fixed worker count, assignment rule and driver hash.
- `responses.jsonl`: merged journal, rebuilt only after workers complete.
- `full/data/*.parquet`, `full/summary.json`, `full/grading_audit.jsonl`: final outputs.
- `full/provenance.json`: original experiment metadata plus the four-worker protocol.

Upload replaces the destination's default train split, preserves original files, and
refuses to overwrite a newer Hub commit. After a successful upload, a NEW run must
set SOURCE_REVISION to the original cleaned commit printed in section 2; the now
annotated `main` is intentionally rejected as input. Existing runs resume their pinned input.

Generation is parallel but exact 4x speedup is not guaranteed. Per-problem seeds remain
unchanged; different GPU scheduling can change exact sampled text. Settings preserve
the original 16-sequence limit per GPU; benchmark before raising it in a new run.